In [4]:
import os
import hashlib
import pandas as pd

current_dir = os.getcwd()
base_dir = os.path.abspath(os.path.join(current_dir, ".."))

RAW_DIR = os.path.join(base_dir, "data", "raw")
TAB_DIR = os.path.join(base_dir, "outputs", "tables")

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

RAW_XLSX = os.path.join(RAW_DIR, "Sample - Superstore.xlsx")

print("读取原始数据")
if not os.path.exists(RAW_XLSX):
    raise FileNotFoundError(f"未找到文件：{RAW_XLSX}")

xl = pd.ExcelFile(RAW_XLSX)

df = xl.parse("Orders")

expect_cols = ["Order ID", "Order Date", "Ship Date", "Ship Mode", "Customer ID",
               "Customer Name", "Segment", "Country", "City", "State", "Postal Code",
               "Region", "Product ID", "Category", "Sub-Category", "Product Name",
               "Sales", "Quantity", "Discount", "Profit"]
missing = [c for c in expect_cols if c not in df.columns]
assert not missing, f"字段缺失：{missing}"
assert len(df) >= 500, f"数据规模不足 500 条，当前 {len(df)} 条"
print(f"  行数 {len(df):,} × 列数 {df.shape[1]}，字段校验通过")

out_raw_csv = os.path.join(RAW_DIR, "orders_raw.csv")
df.to_csv(out_raw_csv, index=False, encoding="utf-8-sig")
print(f"  raw ：orders_raw.csv")

overview = pd.DataFrame([
    {"指标": "记录数（行）", "取值": len(df)},
    {"指标": "字段数（列）", "取值": df.shape[1]},
    {"指标": "订单数（Order ID 去重）", "取值": df["Order ID"].nunique()},
    {"指标": "客户数（Customer ID 去重）", "取值": df["Customer ID"].nunique()},
    {"指标": "商品数（Product ID 去重）", "取值": df["Product ID"].nunique()},
    {"指标": "时间跨度", "取值": f"{pd.to_datetime(df['Order Date']).min():%Y-%m-%d} ~ {pd.to_datetime(df['Order Date']).max():%Y-%m-%d}"},
    {"指标": "销售总额（元）", "取值": round(df["Sales"].sum(), 2)},
    {"指标": "利润总额（元）", "取值": round(df["Profit"].sum(), 2)},
    {"指标": "国家/地区", "取值": "United States（4 个大区、49 个州）"},
])
overview_path = os.path.join(TAB_DIR, "t01_data_overview.csv")
overview.to_csv(overview_path, index=False, encoding="utf-8-sig")
print(f"  t01表已生成 {overview_path}")

读取原始数据
  行数 9,994 × 列数 21，字段校验通过
  raw ：orders_raw.csv
  t01表已生成 C:\Users\zhou\Untitled Folder\outputs\tables\t01_data_overview.csv


D:\anaconda\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
